# Tu copia personal · Aula UTP

Este cuaderno se abre desde GitHub. En Colab elige **Archivo → Guardar una copia en Drive** antes de conservar cambios. Renombra esa copia y ejecuta de arriba abajo. Cada estudiante trabaja en su propio archivo y entorno.

Los ejemplos son didácticos: sigue la sección indicada por Cristian para esta clase. Las comprobaciones son voluntarias, sin entregas ni notas.

# Lectura guiada · Machine Learning UTP
Cristian Alvarado · Ejecuta de arriba hacia abajo. No hay tareas ni entregas. Los ejemplos se revisan con el docente.

## 1. Logística: probabilidad y umbral
Entrada: índice hídrico simulado. Etiqueta: alto rendimiento (1) o no (0). La curva muestra probabilidad estimada; el umbral convierte puntuación en clase. Aquí las métricas se calculan sobre entrenamiento y no prueban generalización. El laboratorio complementario muestra evaluación con datos reservados.

In [ ]:
# Regresión logística — versión mínima y clara (1 sola variable, 1 sola figura)
# -----------------------------------------------------------------------------
# Objetivo didáctico:
# - Que vean la "S" (sigmoide) y entiendan que la salida es PROBABILIDAD de clase.
# - Con 1 variable (índice hídrico) para no distraer: si hay buen balance hídrico,
#   aumenta la probabilidad de “alto rendimiento”.
#
# Copia esta ÚNICA CELDA en Colab y ejecútala.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# 1) Dataset MUY simple (1 variable) ------------------------------------------
# Índice hídrico (mm): positivo = agua suficiente; negativo = déficit.
rng = np.random.default_rng(0)
n = 200
indice_hidrico = rng.normal(loc=0.0, scale=25.0, size=n)  # media 0 para tener casos positivos y negativos

# Probabilidad “verdadera” pedagógica (sigmoide con umbral ~0)
# Ajuste suave: a mayor índice, mayor probabilidad de alto rendimiento.
p_true = 1 / (1 + np.exp(-(indice_hidrico / 8.0)))  # dividir por 8 ensancha la S para verla mejor

# Etiqueta binaria a partir de esa probabilidad (simulamos la realidad con azar)
y = (rng.uniform(0, 1, n) < p_true).astype(int)

X = indice_hidrico.reshape(-1, 1)

print("Vista rápida de los datos (primeras 5 filas):")
display(pd.DataFrame({"indice_hidrico_mm": indice_hidrico[:5].round(2), "alto_rendimiento": y[:5]}))

print(f"Prevalencia de clase positiva (alto_rendimiento=1): {y.mean():.3f}")

# 2) Entrenar Regresión Logística ---------------------------------------------
# Sin estandarizar para que el coeficiente se interprete directo en mm.
logit = LogisticRegression(solver="lbfgs")
logit.fit(X, y)

b0 = float(logit.intercept_[0])
b1 = float(logit.coef_[0, 0])
print("\nModelo aprendido (forma canónica de la sigmoide):")
print("p(y=1 | x) = 1 / (1 + exp(-(b0 + b1 * x)))")
print(f"b0 = {b0:.3f}, b1 = {b1:.3f}")
print(f"Odds ratio por +1 mm en índice hídrico: exp(b1) = {np.exp(b1):.3f}")

# 3) Probabilidades de clase y predicción (umbral 0.5) ------------------------
p_hat = logit.predict_proba(X)[:, 1]
y_pred = (p_hat >= 0.5).astype(int)

acc = accuracy_score(y, y_pred)
cm = confusion_matrix(y, y_pred)
print(f"\nAccuracy de entrenamiento con umbral 0.5: {acc:.3f}")
print("Matriz de confusión (filas = real, columnas = predicho):")
print(cm)

# 4) Visualización única: datos y curva sigmoide ------------------------------
# Ordenamos por x para dibujar la sigmoide suave
idx = np.argsort(indice_hidrico)
x_plot = indice_hidrico[idx]
p_plot = p_hat[idx]

plt.figure(figsize=(7,5))
# Puntos crudos (x, y binario) con jitter vertical para ver densidad
y_jitter = y[idx] + rng.normal(0, 0.02, size=n)
plt.scatter(x_plot, y_jitter, alpha=0.4, label="Observaciones (0/1 con jitter)")

# Curva de probabilidad aprendida
plt.plot(x_plot, p_plot, linewidth=2, label="Probabilidad estimada (sigmoide)")

# Línea de decisión (p=0.5 -> b0 + b1*x = 0 -> x = -b0/b1)
if b1 != 0:
    x_threshold = -b0 / b1
    plt.axvline(x_threshold, linestyle="--", label=f"Umbral p=0.5 (x≈{x_threshold:.1f})")

plt.xlabel("Índice hídrico (mm)")
plt.ylabel("Probabilidad de alto rendimiento")
plt.title("Regresión logística: 1 variable, 1 curva, 1 idea")
plt.ylim(-0.1, 1.1)
plt.legend()
plt.show()

# 5) Mini-demostración práctica ------------------------------------------------
# ¿Qué probabilidad predice el modelo para tres escenarios?
escenarios = np.array([[-40.0], [0.0], [+40.0]])  # déficit, neutral, superávit
p_esc = logit.predict_proba(escenarios)[:, 1]
tabla = pd.DataFrame({"indice_hidrico_mm": escenarios.ravel(), "p_alto_rendimiento": p_esc.round(3)})
print("\nProbabilidades para tres escenarios simples:")
display(tabla)

# Explicación concisa en prints (para el estudiante):
print("\nResumen pedagógico:")
print("1) predict_proba estima probabilidades; predict devuelve clases con una regla de decisión.")
print("2) Esa probabilidad sigue una curva en 'S' (sigmoide).")
print("3) Para decidir clase, usamos un UMBRAL (típicamente 0.5).")
print("4) Si mueves el umbral, cambias la sensibilidad vs. especificidad (no mostrado aquí para mantenerlo simple).")
print("5) Odds = p/(1-p). exp(b1) multiplica las odds por +1 mm, no los puntos porcentuales de probabilidad.")
print("   Aquí b1>0 asocia un índice mayor con mayores odds estimadas; no demuestra causalidad.")


## 2. Random Forest: evaluar casos reservados
Dos entradas: índice hídrico y temperatura. Revisa la separación estratificada, las probabilidades y la matriz de confusión en prueba. La importancia de variables pertenece a este modelo y no demuestra causalidad.

In [ ]:
# Random Forest (clasificación binaria) — versión clara y descriptiva para Agronegocios
# ------------------------------------------------------------------------------------
# Objetivo didáctico:
# - Mostrar cómo Random Forest maneja relaciones NO lineales y umbrales naturales.
# - Usaremos 2 variables simples y muy intuitivas:
#     x1 = índice hídrico (mm)       → balance de agua en el ciclo
#     x2 = temperatura media (°C)    → media del ciclo, no calor acumulado
# - La etiqueta "alto_rendimiento" (1/0) sigue una regla no lineal inventada
#   que sirve para mostrar una frontera no lineal. No valida condiciones agro reales
#   ni compara este bosque con la logística del bloque anterior, que usa otros datos.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# =====================================================
# 1) Dataset sintético con relación NO lineal (2D)
# =====================================================
print("1) Generando dataset 2D (no lineal) con etiqueta agro...")

rng = np.random.default_rng(1)
n = 600

# Variables:
x1 = rng.normal(loc=0.0, scale=35.0, size=n).clip(-120, 120)  # índice hídrico (mm)
x2 = rng.normal(loc=22.0, scale=3.0, size=n).clip(14, 30)     # temperatura media (°C)

# Regla no lineal "oculta"
y_true = (
    (x2 > 19.5) & (x2 < 25.5) & (x1 > -20)
).astype(int)

# Ruido didáctico: invertimos etiquetas al azar con probabilidad 0.08
flip = rng.uniform(0, 1, n) < 0.08
y = np.where(flip, 1 - y_true, y_true)

df = pd.DataFrame({"indice_hidrico_mm": x1, "temp_media_c": x2, "alto_rendimiento": y})
print("   Muestra de datos:")
display(df.head(5))
print(f"   Prevalencia clase positiva: {df['alto_rendimiento'].mean():.3f}")

# =====================================================
# 2) Train/Test y entrenamiento del Random Forest
# =====================================================
print("\n2) Particionando y entrenando RandomForestClassifier...")

X = df[["indice_hidrico_mm", "temp_media_c"]].values
y = df["alto_rendimiento"].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=5,
    random_state=42
)
rf.fit(X_tr, y_tr)
print("   Entrenamiento completado.")

# =====================================================
# 3) Probabilidades, predicción y métricas
# =====================================================
print("\n3) Prediciendo en Test, probabilidades y métricas...")

proba = rf.predict_proba(X_te)[:, 1]
y_pred = (proba >= 0.5).astype(int)

acc = accuracy_score(y_te, y_pred)
cm = confusion_matrix(y_te, y_pred)
print(f"   Accuracy (umbral 0.5): {acc:.3f}")
print("   Matriz de confusión (filas=real, columnas=predicho):")
print(cm)
print("\n   Reporte de clasificación:")
print(classification_report(y_te, y_pred, digits=3, zero_division=0))

# =====================================================
# 4) Importancia de variables
# =====================================================
print("\n4) Importancia de variables (Gini):")
feat_names = ["indice_hidrico_mm", "temp_media_c"]
importancias = rf.feature_importances_
imp_df = pd.DataFrame({"variable": feat_names, "importancia": importancias}).sort_values("importancia", ascending=False)
display(imp_df)

# =====================================================
# 5) Visualización 1: Superficie de probabilidad (2D)
# =====================================================
print("\n5) Graficando superficie de probabilidad y puntos de datos...")

x1_min, x1_max = X[:,0].min()-5, X[:,0].max()+5
x2_min, x2_max = X[:,1].min()-1, X[:,1].max()+1

xx1, xx2 = np.meshgrid(
    np.linspace(x1_min, x1_max, 200),
    np.linspace(x2_min, x2_max, 200)
)
grid = np.c_[xx1.ravel(), xx2.ravel()]
grid_proba = rf.predict_proba(grid)[:, 1].reshape(xx1.shape)

plt.figure(figsize=(8,6))
# Mapa de calor de probabilidades
mapa = plt.contourf(xx1, xx2, grid_proba, levels=20, alpha=0.85, cmap="RdYlGn")
# Frontera 0.5
plt.contour(xx1, xx2, grid_proba, levels=[0.5], linewidths=2, colors="k", linestyles="--")
# Puntos de entrenamiento
puntos = plt.scatter(X_tr[:,0], X_tr[:,1], c=y_tr, edgecolor="k", alpha=0.6)
plt.xlabel("Índice hídrico (mm)")
plt.ylabel("Temperatura media (°C)")
plt.title("Random Forest — Probabilidad estimada de alto rendimiento")
clases, _ = puntos.legend_elements()
frontera = Line2D([], [], color="k", linestyle="--", linewidth=2)
plt.legend(handles=[*clases, frontera], labels=["No alto (0)", "Alto (1)", "Frontera p=0.5"], loc="lower left")
plt.colorbar(mapa, label="p(y=1)")
plt.show()

# =====================================================
# 6) Visualización 2: Importancia de variables (barras)
# =====================================================
plt.figure(figsize=(6,4))
order = np.argsort(importancias)[::-1]
plt.bar(np.array(feat_names)[order], importancias[order])
plt.ylabel("Importancia (impureza Gini)")
plt.title("Importancia de variables en el bosque")
plt.show()

# =====================================================
# 7) Mini-demostración práctica: tres escenarios
# =====================================================
print("\n7) Probabilidades en tres escenarios simples:")

escenarios = pd.DataFrame({
    "indice_hidrico_mm": [-60, 0, 70],   # déficit — neutro — superávit
    "temp_media_c":      [18.0, 23.0, 27.0]
})
p_esc = rf.predict_proba(escenarios.values)[:, 1]
tabla = escenarios.copy()
tabla["p_alto_rendimiento"] = np.round(p_esc, 3)
display(tabla)

# =====================================================
# 8) Explicación operativa (imprimible)
# =====================================================
print("\nResumen didáctico:")
print("1) Random Forest entrena muchos árboles de decisión.")
print("2) Cada árbol decide con reglas simples (ej. si temp < 25 y agua > -10).")
print("3) El bosque combina esas reglas → probabilidad promedio.")
print("4) Captura relaciones irregulares y no lineales sin fórmulas explícitas.")
print("5) La importancia Gini resume reducción de impureza; puede tener sesgos y no mide efectos causales.")


## 3. SVM: frontera y margen
Ejemplo sintético para ver separación. Sigue cómo la frontera divide el espacio de entradas; el margen no es una probabilidad. Consulta las etiquetas de los ejes antes de interpretar distancias.

In [ ]:
# ===========================================
# SVM — Hiperplano óptimo (ejemplo simple, didáctico)
# ===========================================

# -----------------------
# 1) IMPORTS Y SETUP
# -----------------------
# numpy: operaciones numéricas
# matplotlib: gráficas 2D
# sklearn: datasets sintéticos y SVM (SVC)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_circles
from sklearn.svm import SVC

# Para reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# -----------------------
# 2) DATOS (CASO LINEAL)
# -----------------------
# make_blobs: genera dos nubes (clases 0 y 1) en R^2
X, y = make_blobs(
    n_samples=120,    # total de puntos
    centers=2,        # número de clases
    cluster_std=1.0,  # dispersión de cada nube
    random_state=RANDOM_STATE
)

print("Forma de X:", X.shape, "| Forma de y:", y.shape)  # (120, 2) y (120,)

# -----------------------
# 3) ENTRENAR SVM LINEAL
# -----------------------
# SVC(kernel='linear') aprende w y b del hiperplano w^T x + b = 0
# C penaliza las violaciones del margen (incluye puntos correctos dentro de él).
# Un C menor permite más violaciones; un C mayor las penaliza más.
# El margen puede cambiar, aunque el efecto puede saturarse en datos separables.
clf = SVC(kernel='linear', C=1.0, random_state=RANDOM_STATE)
clf.fit(X, y)

# w: vector normal del hiperplano | b: término independiente
w = clf.coef_[0]
b = clf.intercept_[0]

# Ancho del margen geométrico: 2 / ||w||
margin_width = 2.0 / np.linalg.norm(w)

print("\n=== MODELO LINEAL C=1.0 ===")
print("w:", w)
print("b:", b)
print("Ancho del margen (2/||w||):", margin_width)
print("N° vectores de soporte:", clf.support_vectors_.shape[0])

# -----------------------
# 4) FUNCIÓN DE PLOTEO
# -----------------------
# Dibuja puntos, hiperplano (nivel 0), márgenes (niveles -1 y +1)
# y marca los vectores de soporte (puntos con círculo sin relleno).
def plot_svm_2d(clf, X, y, title="SVM — hiperplano y márgenes"):
    # Rango con acolchado para ver bien la frontera
    x1_min, x1_max = X[:,0].min() - 1.0, X[:,0].max() + 1.0
    x2_min, x2_max = X[:,1].min() - 1.0, X[:,1].max() + 1.0

    # Grilla densa para evaluar la función de decisión
    xx1, xx2 = np.meshgrid(
        np.linspace(x1_min, x1_max, 300),
        np.linspace(x2_min, x2_max, 300)
    )
    grid = np.c_[xx1.ravel(), xx2.ravel()]
    Z = clf.decision_function(grid).reshape(xx1.shape)  # signo -> clase; en el caso lineal, dividir por ||w|| da distancia geométrica con signo

    # Puntos coloreados por clase (paleta por defecto)
    plt.figure()
    plt.scatter(X[:,0], X[:,1], c=y)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)

    # Contornos en -1, 0, +1: márgenes e hiperplano
    plt.contour(xx1, xx2, Z, levels=[-1, 0, 1], linestyles=["--", "-", "--"], linewidths=1.5)

    # Vectores de soporte: pueden estar sobre el margen o dentro de él, incluso mal clasificados
    sv = clf.support_vectors_
    plt.scatter(sv[:,0], sv[:,1], s=120, facecolors='none', linewidths=1.5)

    plt.show()

# Visualización del caso lineal
plot_svm_2d(clf, X, y, title="SVM lineal (C=1.0) — hiperplano y márgenes")

# -----------------------
# 5) EFECTO DEL PARÁMETRO C
# -----------------------
# En estos datos, C=0.01 y C=100 sí producen márgenes distintos.
# C=0.1, C=1 y C=100 dan el mismo margen: aumentar C no siempre cambia la solución.
for C in [0.01, 100]:
    clf_C = SVC(kernel='linear', C=C, random_state=RANDOM_STATE)
    clf_C.fit(X, y)
    wC = clf_C.coef_[0]
    marginC = 2.0 / np.linalg.norm(wC)
    print(f"\n=== EFECTO C={C} ===")
    print(f"margen ~ {marginC:.3f} | #soportes={clf_C.support_vectors_.shape[0]}")
    plot_svm_2d(clf_C, X, y, title=f"SVM lineal (C={C}) — efecto sobre el margen")

# -----------------------
# 6) CASO NO LINEAL (KERNEL RBF)
# -----------------------
# make_circles crea dos anillos concéntricos (no separables linealmente).
# Kernel RBF equivale a proyectar a un espacio de mayor dimensión donde
# sí existe una separación lineal (la frontera aparece curva en 2D).
X_nl, y_nl = make_circles(
    n_samples=200, factor=0.4, noise=0.1, random_state=RANDOM_STATE
)

clf_rbf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=RANDOM_STATE)
clf_rbf.fit(X_nl, y_nl)

plot_svm_2d(clf_rbf, X_nl, y_nl, title="SVM con kernel RBF — frontera no lineal")

# -----------------------
# 7) RESUMEN RÁPIDO (impreso)
# -----------------------
print("\nResumen:")
print("- SVM busca el hiperplano con margen máximo entre clases.")
print("- Los vectores de soporte definen la solución; pueden estar en o dentro de los márgenes.")
print("- C penaliza violaciones del margen; cambiarlo puede cambiar la solución, pero no siempre.")
print("- Con RBF la frontera es no lineal; decision_function no es distancia directa en el plano original.")


## 4. KNN: vecinos de una flor
Este bloque usa Iris, no café: las entradas son medidas de flores y la salida es su especie. Comprueba escalado, k y evaluación. La cercanía depende de las variables y de su escala.

In [ ]:
# ===========================================
# k-NN — Ejemplo simple y real (Iris dataset)
# Objetivo: dada una flor con medidas reales, predecir su especie
# usando "vecinos más cercanos" (similitud en el espacio de features).
# ===========================================

# 1) IMPORTS
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 2) DATOS REALES: IRIS
#   - 150 flores reales medidas en cm.
#   - 3 especies: setosa, versicolor, virginica.
#   - Usaremos 2 features para poder visualizar: largo y ancho del pétalo.
iris = load_iris()
X = iris.data[:, 2:4]                  # [petal length, petal width]
y = iris.target                        # 0=setosa, 1=versicolor, 2=virginica
feature_names = iris.feature_names[2:4]
class_names = iris.target_names

print("X shape:", X.shape, "| y shape:", y.shape)
print("Features:", feature_names)
print("Clases:", class_names)

# 3) SPLIT TRAIN/TEST
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# 4) ELEGIR k (número de vecinos) POR VALIDACIÓN CRUZADA
#    Probamos varios k. En tres clases, k impar no elimina todos los empates (ej. 2-2-1).
candidatos_k = [1, 3, 5, 7, 9, 11, 13]
mejor_k, mejor_score = None, -1

for k in candidatos_k:
    # Pipeline: escalado (recomendable en kNN) + clasificador kNN
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])
    # 5-fold CV sobre el set de entrenamiento
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5)
    score = cv_scores.mean()
    print(f"k={k} | CV mean acc={score:.3f}")
    if score > mejor_score:
        mejor_score, mejor_k = score, k

print(f"\nMejor k según CV: {mejor_k} (acc≈{mejor_score:.3f})")

# 5) ENTRENAR MODELO FINAL CON k ÓPTIMO
modelo = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=mejor_k))
])
modelo.fit(X_train, y_train)

# 6) EVALUAR EN TEST
y_pred = modelo.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\nAccuracy en test: {acc:.3f}")
print("\nMatriz de confusión:\n", confusion_matrix(y_test, y_pred))
print("\nReporte de clasificación:\n", classification_report(y_test, y_pred, target_names=class_names))

# 7) VISUALIZAR REGIONES DE DECISIÓN (solo 2D)
#    Creamos una malla en el plano [petal length, petal width] y pintamos
#    las regiones donde el modelo predice cada especie.
def plot_decision_regions(modelo, X, y, title="k-NN — Regiones de decisión (Iris)"):
    # Usamos el modelo ya entrenado: dibujar no modifica sus pesos ni su escalado.

    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 400),
        np.linspace(y_min, y_max, 400)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = modelo.predict(grid).reshape(xx.shape)

    # Mapa de decisión (sin fijar colores manualmente)
    plt.figure()
    plt.contourf(xx, yy, Z, alpha=0.25)  # regiones
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor="k")  # puntos reales
    plt.xlabel(feature_names[0])
    plt.ylabel(feature_names[1])
    plt.title(title)
    plt.show()

plot_decision_regions(modelo, X, y, title=f"k-NN (k={mejor_k}) — Regiones de decisión con Iris")

# 8) PREDICCIÓN PARA UNA NUEVA FLOR HIPOTÉTICA
#    Medidas elegidas para demostrar el uso; no se realizó una medición nueva:
#    pétalo largo=4.8 cm, pétalo ancho=1.7 cm
nueva_flor = np.array([[4.8, 1.7]])
pred = modelo.predict(nueva_flor)[0]
proba_vecinos = modelo.named_steps["knn"].kneighbors(
    modelo.named_steps["scaler"].transform(nueva_flor), n_neighbors=mejor_k, return_distance=False
)

print("\nNueva muestra:", nueva_flor.tolist())
print("Predicción de especie:", class_names[pred])
print(f"Vecinos usados (posiciones dentro del conjunto de entrenamiento): {proba_vecinos[0].tolist()}")

# ============================
# RESUMEN DIDÁCTICO:
# - Problema real: clasificar especie de flor con medidas físicas.
# - k-NN: “vota” con las k flores más parecidas (distancia Euclídea).
# - Escalar ayuda porque las distancias son sensibles a la escala.
# - k se ajusta con CV para equilibrar sesgo/varianza.
# ============================
